In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from bayes_opt import BayesianOptimization
import warnings
import random

warnings.filterwarnings('ignore')

random_state = 1
np.random.seed(random_state)
random.seed(random_state)

def fill_nan_with_median(data, name="数据"):
    """使用各列的中位数（数值列）或众数/默认值（非数值列）填充NaN"""
    if data.isnull().values.any():
        nan_count = data.isnull().sum().sum()
        print(f"警告：{name}包含 {nan_count} 个NaN值，将使用中位数/众数进行填充")
        
        num_cols = data.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            if data[col].isnull().any():
                median_val = data[col].median()
                if pd.isna(median_val):
                    median_val = 0.0
                data[col] = data[col].fillna(median_val)
                
        non_num_cols = data.select_dtypes(exclude=[np.number]).columns
        for col in non_num_cols:
            if data[col].isnull().any():
                mode_val = data[col].mode()
                fill_val = mode_val[0] if not mode_val.empty else 'missing'
                data[col] = data[col].fillna(fill_val)
                
        print(f"{name}：共填充了 {nan_count} 个NaN值")
    return data


def CV_group(data):
    """负载均衡的数据分组策略，将数据按ID均匀分配到5个组"""
    data = data.copy()
    data = fill_nan_with_median(data, "CV_group输入数据")
    
    data['ref_FO'] = data['ID'].map(str)
    ref_JQ = data['ID']
    tt_ion = pd.DataFrame(ref_JQ.value_counts(ascending=False))
    tt_ion_index = tt_ion.index

    one, two, three, four, five = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for i in range(len(tt_ion_index)):
        a = [len(one), len(two), len(three), len(four), len(five)]
        min_idx = a.index(min(a))
        current_data = data[data['ID'] == tt_ion_index[i]]
        
        if min_idx == 0: one = pd.concat([one, current_data])
        elif min_idx == 1: two = pd.concat([two, current_data])
        elif min_idx == 2: three = pd.concat([three, current_data])
        elif min_idx == 3: four = pd.concat([four, current_data])
        elif min_idx == 4: five = pd.concat([five, current_data])

    print(f"分组长度: one={len(one)}, two={len(two)}, three={len(three)}, four={len(four)}, five={len(five)}")
    return one, two, three, four, five


class ML_model():
    def __init__(self, train, test, one, two, three, four, five, features):
        self.features = features
        self.scaler_X = StandardScaler()
        
        for name, df in [("训练集", train), ("测试集", test), ("分组1", one), ("分组2", two), 
                         ("分组3", three), ("分组4", four), ("分组5", five)]:
            self._check_and_fill_data(df, name)
        
        self.X_train = pd.DataFrame(train)
        self.X_test = pd.DataFrame(test)
        
        if 'Jw' not in train.columns or 'Jw' not in test.columns:
            raise ValueError("数据必须包含'Jw'列作为目标变量")
        
        self.y_train = train['Jw']
        self.y_test = test['Jw']
        
        self.X_train_1 = pd.concat([two, three, four, five], join='outer')
        self.X_test_1 = one.copy()
        self.X_train_2 = pd.concat([one, three, four, five], join='outer')
        self.X_test_2 = two.copy()
        self.X_train_3 = pd.concat([one, two, four, five], join='outer')
        self.X_test_3 = three.copy()
        self.X_train_4 = pd.concat([one, two, three, five], join='outer')
        self.X_test_4 = four.copy()
        self.X_train_5 = pd.concat([one, two, three, four], join='outer')
        self.X_test_5 = five.copy()
        
        self._clean_cv_datasets()
        
        for i in range(1, 6):
            setattr(self, f'y_train_{i}', self._safe_get_target(getattr(self, f'X_train_{i}'), 'Jw', f'分组{i}训练集'))
            setattr(self, f'y_test_{i}', self._safe_get_target(getattr(self, f'X_test_{i}'), 'Jw', f'分组{i}测试集'))
            setattr(self, f'X_train_{i}', self._safe_select_features(getattr(self, f'X_train_{i}'), f"分组{i}训练集"))
            setattr(self, f'X_test_{i}', self._safe_select_features(getattr(self, f'X_test_{i}'), f"分组{i}测试集"))
            
        print("数据初始化完成，所有NaN问题已处理")
    
    def _check_and_fill_data(self, data, name):
        if data is None or len(data) == 0: return
        if data.isnull().sum().sum() > 0:
            fill_nan_with_median(data, name)
    
    def _clean_cv_datasets(self):
        for i in range(1, 6):
            for prefix, name in [('X_train', '训练集'), ('X_test', '测试集')]:
                df = getattr(self, f'{prefix}_{i}')
                if df is not None and df.isnull().sum().sum() > 0:
                    fill_nan_with_median(df, f"{name}{i}")
    
    def _safe_get_target(self, data, target_col, name):
        if target_col not in data.columns:
            raise ValueError(f"{name} 缺少目标列 '{target_col}'")
        y = data[target_col].copy()
        if y.isnull().any():
            y = y.fillna(y.median())
        return y
    
    def _safe_select_features(self, data, name):
        X = data[self.features].copy()
        if X.isnull().values.any():
            for col in X.columns:
                X[col] = X[col].fillna(X[col].median())
        if np.isinf(X.values).any():
            X = X.replace([np.inf, -np.inf], np.nan)
            for col in X.columns:
                X[col] = X[col].fillna(X[col].median())
        return X
    
    def rem_name(self, org_data):
        if 'ref_ID' not in org_data.columns:
            return pd.DataFrame(index=org_data.index)
        return org_data.loc[:, ['ref_ID']].copy()
    
    def CV_train(self, params, X_train, y_train, X_test, y_test):
        """交叉验证训练方法 (KNN版本)"""
        try:
            if X_train.empty or X_test.empty or len(y_train) == 0 or len(y_test) == 0:
                return float('inf')
            
            X_train_np = np.nan_to_num(X_train.values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_train_np = y_train.values.astype(np.float32)
            X_test_np = np.nan_to_num(X_test.values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_test_np = y_test.values.astype(np.float32)
            
            scaler_X = StandardScaler()
            X_train_scaled = scaler_X.fit_transform(X_train_np)
            X_test_scaled = scaler_X.transform(X_test_np)
            
            n_neighbors = int(round(params['n_neighbors']))
            p = int(round(params['p']))
            weights = 'distance' if params['weights'] > 0.5 else 'uniform'
            
            model = KNeighborsRegressor(
                n_neighbors=n_neighbors,
                weights=weights,
                p=p,
                algorithm='auto',
                n_jobs=-1
            )
            
            model.fit(X_train_scaled, y_train_np)
            y_pred = model.predict(X_test_scaled)
            
            y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_test_clean = np.nan_to_num(y_test_np, nan=0.0, posinf=0.0, neginf=0.0)
            
            rmse = np.sqrt(mean_squared_error(y_test_clean, y_pred))
            return rmse
            
        except Exception as e:
            print(f"KNN训练过程中出错: {str(e)}")
            return float('inf')
    
    def data_prep_for_model(self, params):
        """执行五折交叉验证"""
        fold_results = [
            self.CV_train(params, getattr(self, f'X_train_{i}'), getattr(self, f'y_train_{i}'), 
                          getattr(self, f'X_test_{i}'), getattr(self, f'y_test_{i}'))
            for i in range(1, 6)
        ]
        
        valid_rmses = [rmse for rmse in fold_results if not np.isinf(rmse) and not np.isnan(rmse)]
        if len(valid_rmses) == 0: return -1
        
        rmse_mean = np.mean(valid_rmses)
        print(f"五折RMSE: {[round(r, 4) for r in fold_results]} | 平均RMSE: {rmse_mean:.4f}")
        return -1 * rmse_mean
    
    def knn_evaluate(self, n_neighbors, weights, p):
        """KNN超参数评估函数"""
        params = {'n_neighbors': n_neighbors, 'weights': weights, 'p': p}
        return self.data_prep_for_model(params)
    
    def param(self):
        """执行贝叶斯优化超参数搜索"""
        pbounds = {
            'n_neighbors': (1, 50),
            'weights': (0, 1),
            'p': (1, 2)
        }
        
        bo = BayesianOptimization(
            f=self.knn_evaluate,
            pbounds=pbounds,
            verbose=2,
            random_state=random_state
        )
        
        print('=' * 20)
        print('Starting Bayesian Optimization for KNN...')
        print('=' * 20)
        
        try:
            bo.maximize(init_points=10, n_iter=300)
        except Exception as e:
            print(f"贝叶斯优化出错: {str(e)}")
            return {'n_neighbors': 5, 'weights': 0.0, 'p': 2}
        
        best_params = bo.max['params']
        print(f"最佳超参数: {best_params}")
        return best_params
    
    def final_model(self, X_train, y_train):
        """构建最终的 KNN 模型"""
        try:
            params = self.param()
            
            X_train_np = np.nan_to_num(X_train[self.features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_train_np = y_train.values.astype(np.float32)
            
            self.scaler_X.fit(X_train_np)
            X_train_scaled = self.scaler_X.transform(X_train_np)
            
            n_neighbors = int(round(params['n_neighbors']))
            p = int(round(params['p']))
            weights = 'distance' if params['weights'] > 0.5 else 'uniform'
            
            print("\n训练最终的 KNN 模型...")
            model = KNeighborsRegressor(
                n_neighbors=n_neighbors,
                weights=weights,
                p=p,
                algorithm='auto',
                n_jobs=-1
            )
            model.fit(X_train_scaled, y_train_np)
            print("最终模型训练完成！")
            return model
            
        except Exception as e:
            print(f"训练最终模型时出错: {str(e)}，使用默认参数...")
            model = KNeighborsRegressor(n_neighbors=5, weights='uniform', p=2)
            model.fit(self.scaler_X.transform(X_train_np), y_train.values.astype(np.float32))
            return model
    
    def result(self, model):
        """模型性能评估"""
        try:
            X_train_name = self.rem_name(self.X_train)
            X_test_name = self.rem_name(self.X_test)
            
            y_train = self.y_train
            y_test = self.y_test
            
            X_train_np = np.nan_to_num(self.X_train[self.features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            X_test_np = np.nan_to_num(self.X_test[self.features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            
            X_train_scaled = self.scaler_X.transform(X_train_np)
            X_test_scaled = self.scaler_X.transform(X_test_np)
            
            y_train_pred = model.predict(X_train_scaled)
            y_pred = model.predict(X_test_scaled)
            
            y_train_pred = np.nan_to_num(y_train_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_train_clean = np.nan_to_num(y_train.values, nan=0.0, posinf=0.0, neginf=0.0)
            y_test_clean = np.nan_to_num(y_test.values, nan=0.0, posinf=0.0, neginf=0.0)
            
            train_r2, train_rmse, train_mae = r2_score(y_train_clean, y_train_pred), np.sqrt(mean_squared_error(y_train_clean, y_train_pred)), mean_absolute_error(y_train_clean, y_train_pred)
            test_r2, test_rmse, test_mae = r2_score(y_test_clean, y_pred), np.sqrt(mean_squared_error(y_test_clean, y_pred)), mean_absolute_error(y_test_clean, y_pred)
            
            print("\n" + "=" * 60)
            print("模型性能结果")
            print("=" * 60)
            print(f"【训练集】 R²: {train_r2:.4f} | RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f}")
            print(f"【验证集】 R²: {test_r2:.4f} | RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f}")
            print("=" * 60)
            
            y_pred_df = pd.DataFrame(y_pred, columns=['pred'], index=self.X_test.index)
            y_test_df = pd.DataFrame(y_test_clean, columns=['y_test'], index=self.X_test.index)
            y_train_pred_df = pd.DataFrame(y_train_pred, columns=['pred'], index=self.X_train.index)
            
            total = pd.concat([X_train_name, self.X_train[self.features], pd.Series(y_train_clean, name='Jw', index=self.X_train.index)], axis=1)
            total_train = pd.concat([total, y_train_pred_df], axis=1)
            
            df_1_test = pd.concat([self.X_test[self.features], y_test_df], axis=1)
            total_test = pd.concat([X_test_name, df_1_test, y_pred_df], axis=1)
            
            total_train['error'] = total_train['Jw'] - total_train['pred']
            total_test['error'] = total_test['y_test'] - total_test['pred']
            
            return self, total_train, total_test, total_train[['Jw', 'pred', 'error']], total_test[['y_test', 'pred', 'error']]
            
        except Exception as e:
            print(f"评估过程中出错: {str(e)}")
            return self, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()


try:
    df = pd.read_excel('FO xgboost.xlsx', sheet_name='data')
    print("成功加载Excel文件")
except Exception as e:
    print(f"加载Excel文件失败: {str(e)}，使用示例数据...")
    np.random.seed(42)
    n_samples = 200
    df = pd.DataFrame({
        'ID': np.random.randint(1, 10, n_samples),
        'MPDC': np.random.normal(0, 1, n_samples), 'TMCC': np.random.normal(0, 1, n_samples),
        'MPDIPT': np.random.normal(0, 1, n_samples), 'TMCIPT': np.random.normal(0, 1, n_samples),
        'MT': np.random.normal(0, 1, n_samples), 'PTE': np.random.normal(0, 1, n_samples),
        'PTI': np.random.normal(0, 1, n_samples), 'OPS': np.random.normal(0, 1, n_samples),
        'CT': np.random.normal(0, 1, n_samples), 'WCA': np.random.normal(0, 1, n_samples),
        'WP': np.random.normal(0, 1, n_samples), 'MO': np.random.normal(0, 1, n_samples),
        'DST': np.random.normal(0, 1, n_samples), 'DSC': np.random.normal(0, 1, n_samples),
        'ST': np.random.normal(0, 1, n_samples), 'CV': np.random.normal(0, 1, n_samples),
        'Jw': np.random.normal(50, 20, n_samples), 'Js': np.random.normal(0, 1, n_samples),
        'R': np.random.normal(50, 20, n_samples), 'ref_ID': np.random.randint(1, 5, n_samples)
    })

expected_columns = ['ID', 'MPDC', 'TMCC', 'MPDIPT', 'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS',
                  'CT', 'WCA', 'WP', 'MO', 'DST', 'DSC', 'ST', 'CV', 'Jw', 'Js', 'R', 'ref_ID']

if len(df.columns) == len(expected_columns):
    df.columns = expected_columns

df.index = range(len(df))
features = ['MPDC', 'TMCC', 'MPDIPT', 'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS',
            'CT', 'WCA', 'WP', 'MO', 'DST', 'DSC', 'ST', 'CV']
features = [col for col in features if col in df.columns]

if 'ref_ID' in df.columns:
    ref_id = pd.to_numeric(df['ref_ID'], errors='coerce')
    train = df[ref_id != 1].copy()
    test = df[ref_id == 1].copy()
    if len(test) == 0:
        from sklearn.model_selection import train_test_split
        train, test = train_test_split(df, test_size=0.2, random_state=42)
else:
    from sklearn.model_selection import train_test_split
    train, test = train_test_split(df, test_size=0.2, random_state=42)

print(f"训练集大小: {len(train)}, 测试集大小: {len(test)}")

df = fill_nan_with_median(df, "全局数据集df")

try:
    one, two, three, four, five = CV_group(train.copy())
except Exception as e:
    print(f"数据分组出错: {str(e)}，使用随机分组...")
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(kf.split(train))
    one, two, three, four, five = [train.iloc[splits[i][1]] for i in range(5)]

ml_model = ML_model(train, test, one, two, three, four, five, features)
final_model = ml_model.final_model(train, train['Jw'])
result, total_train, total_test, train_error, test_error = ml_model.result(final_model)

try:
    import plotly.express as px
    import plotly.graph_objects as go
    
    if not total_train.empty and not total_test.empty:
        total_train['mark'] = 'train'
        total_test['mark'] = 'test'
        total_data = pd.concat([total_train, total_test])
    
        X_title, Y_title = 'Real R (%)', 'Pred R (%)'
        fig = px.scatter(total_data, x='Jw', y='pred', color='mark', width=700, height=600,
                         labels={'Jw': X_title, 'pred': Y_title},
                         color_discrete_map={'train': '1f77b4', 'test': 'ff7f0e'})
    
        fig.update_layout(
            xaxis=dict(title_text=X_title, title_font={"size": 25}, tickfont=dict(size=18), title_standoff=20, ticks="inside"),
            yaxis=dict(title_text=Y_title, title_font={"size": 25}, tickfont=dict(size=18), title_standoff=20, ticks="inside"),
            plot_bgcolor='rgb(256,256,256)', showlegend=True
        )
        fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
        fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
    
        min_val = min(total_data['Jw'].min(), total_data['pred'].min())
        max_val = max(total_data['Jw'].max(), total_data['pred'].max())
        fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], 
                                 line=dict(color='red', width=2, dash='dash'), name='Perfect Prediction'))
    
        fig.update_traces(marker=dict(size=8, opacity=0.7, line=dict(width=0.5)), selector=dict(mode='markers'))
        fig.show()
except Exception as e:
    print(f"生成可视化图表出错: {str(e)}")

print("\n程序执行完成！")